<a href="https://colab.research.google.com/github/MohammedShahad7/Data-Science-/blob/main/Exploring%20Patterns%20in%20Real-World%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==========================================
# DATASET GENERATION SCRIPT
# Run this cell once to create the dataset
# ==========================================

def generate_urbankart_data(num_rows=2000):
    np.random.seed(42)
    random.seed(42)

    # 1. Date Generation (Over 2 years)
    start_date = datetime(2023, 1, 1)
    dates = [start_date + timedelta(days=np.random.randint(0, 730)) for _ in range(num_rows)]
    dates.sort()

    # 2. Categorical Data
    regions = ['North', 'South', 'East', 'West', 'North', 'South', 'east'] # 'east' is an inconsistency
    categories = ['Electronics', 'Fashion', 'Home & Kitchen', 'Books', 'Beauty']
    segments = ['Consumer', 'Corporate', 'Home Office']
    payment_modes = ['Card', 'UPI', 'COD', 'Card', 'UPI', 'cash'] # 'cash' is an inconsistency vs 'COD'

    data = {
        'Order_ID': [f'UK-{10000+i}' for i in range(num_rows)],
        'Order_Date': dates,
        'Region': [random.choice(regions) for _ in range(num_rows)],
        'Category': [random.choice(categories) for _ in range(num_rows)],
        'Customer_Segment': [random.choice(segments) for _ in range(num_rows)],
        'Payment_Mode': [random.choice(payment_modes) for _ in range(num_rows)],
        'Delivery_Days': np.random.randint(1, 10, num_rows).astype(float)
    }

    df = pd.DataFrame(data)

    # 3. Numerical Data with Relationships & Noise
    # Base Sales
    df['Sales'] = np.random.lognormal(mean=4.5, sigma=1.2, size=num_rows)

    # Discount (0% to 80%)
    df['Discount'] = np.random.choice([0, 0.1, 0.2, 0.3, 0.5, 0.8], size=num_rows, p=[0.4, 0.2, 0.15, 0.15, 0.05, 0.05])

    # Profit Calculation (Sales - Cost) + Noise
    # Profit is roughly 20% of sales, minus discount impact, plus random variation
    df['Profit'] = (df['Sales'] * 0.2) - (df['Sales'] * df['Discount']) + np.random.normal(0, 20, num_rows)

    # Customer Rating (1 to 5) - correlated weakly with delivery days
    # If delivery > 7 days, rating likely lower
    ratings = []
    for days in df['Delivery_Days']:
        if days > 7:
            ratings.append(np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2]))
        else:
            ratings.append(np.random.choice([3, 4, 5], p=[0.1, 0.3, 0.6]))
    df['Customer_Rating'] = ratings

    # 4. Injecting Data Quality Issues

    # Missing Values
    # Set 5% of Ratings to NaN
    df.loc[df.sample(frac=0.05).index, 'Customer_Rating'] = np.nan
    # Set 2% of Delivery Days to NaN
    df.loc[df.sample(frac=0.02).index, 'Delivery_Days'] = np.nan

    # Outliers
    # Create 5 massive sales orders
    outlier_indices = np.random.choice(df.index, 5, replace=False)
    df.loc[outlier_indices, 'Sales'] = df.loc[outlier_indices, 'Sales'] * 50

    # Create some negative profit outliers (extreme loss)
    loss_indices = np.random.choice(df.index, 5, replace=False)
    df.loc[loss_indices, 'Profit'] = -2000

    # Rounding
    df['Sales'] = df['Sales'].round(2)
    df['Profit'] = df['Profit'].round(2)

    return df

# Generate and Save
df_urban = generate_urbankart_data()
df_urban.to_csv('urbankart_transactions.csv', index=False)

print("✅ Dataset 'urbankart_transactions.csv' generated successfully!")
print(f"Row count: {len(df_urban)}")
print("Columns:", list(df_urban.columns))
print("\nFirst 5 rows:")
print(df_urban.head())

✅ Dataset 'urbankart_transactions.csv' generated successfully!
Row count: 2000
Columns: ['Order_ID', 'Order_Date', 'Region', 'Category', 'Customer_Segment', 'Payment_Mode', 'Delivery_Days', 'Sales', 'Discount', 'Profit', 'Customer_Rating']

First 5 rows:
   Order_ID Order_Date Region Category Customer_Segment Payment_Mode  \
0  UK-10000 2023-01-01  South   Beauty         Consumer         Card   
1  UK-10001 2023-01-01  North  Fashion        Corporate         cash   
2  UK-10002 2023-01-01  North    Books         Consumer         Card   
3  UK-10003 2023-01-01  South    Books        Corporate         Card   
4  UK-10004 2023-01-01   East    Books      Home Office          UPI   

   Delivery_Days   Sales  Discount  Profit  Customer_Rating  
0            2.0  171.74       0.0   65.11              4.0  
1            7.0  112.29       0.1   20.84              4.0  
2            6.0   50.66       0.0   56.55              5.0  
3            4.0   10.72       0.2   12.07              5.0  
4 

In [2]:
df_urban.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Order_ID          2000 non-null   object        
 1   Order_Date        2000 non-null   datetime64[ns]
 2   Region            2000 non-null   object        
 3   Category          2000 non-null   object        
 4   Customer_Segment  2000 non-null   object        
 5   Payment_Mode      2000 non-null   object        
 6   Delivery_Days     1960 non-null   float64       
 7   Sales             2000 non-null   float64       
 8   Discount          2000 non-null   float64       
 9   Profit            2000 non-null   float64       
 10  Customer_Rating   1900 non-null   float64       
dtypes: datetime64[ns](1), float64(5), object(5)
memory usage: 172.0+ KB


In [3]:
df_urban.shape

(2000, 11)

In [6]:
df_urban.dtypes

,0
Order_ID,object
Order_Date,datetime64[ns]
Region,object
Category,object
Customer_Segment,object
Payment_Mode,object
Delivery_Days,float64
Sales,float64
Discount,float64
Profit,float64


In [7]:
df_urban.isnull().sum()

,0
Order_ID,0
Order_Date,0
Region,0
Category,0
Customer_Segment,0
Payment_Mode,0
Delivery_Days,40
Sales,0
Discount,0
Profit,0


In [8]:
numeric_col=df_urban.select_dtypes(include=['int64','float64']).columns

for col in numeric_col:
    df_urban[col].fillna(df_urban[col].median(),inplace=True)

/tmp/ipykernel_2845/3186413214.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_urban[col].fillna(df_urban[col].median(),inplace=True)


In [27]:
categerical_col=df_urban.select_dtypes(include=['object']).columns

for col in categerical_col:
  df_urban[col].fillna(df_urban[col].mode()[0],inplace=True)



/tmp/ipykernel_2845/166116820.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_urban[col].fillna(df_urban[col].mode()[0],inplace=True)


In [11]:
df_urban.isnull().sum()

,0
Order_ID,0
Order_Date,0
Region,0
Category,0
Customer_Segment,0
Payment_Mode,0
Delivery_Days,0
Sales,0
Discount,0
Profit,0


In [13]:
df_urban['Region'].unique()

array(['South', 'North', 'East', 'West', 'east'], dtype=object)

In [14]:
df_urban['Payment_Mode'].unique()

array(['Card', 'cash', 'UPI', 'COD'], dtype=object)

In [15]:
df_urban['Region']=df_urban['Region'].str.strip().str.title()
df_urban['Payment_Mode']=df_urban['Payment_Mode'].str.strip().str.title()

In [16]:
df_urban['Region'].unique()

array(['South', 'North', 'East', 'West'], dtype=object)

In [17]:
df_urban['Payment_Mode'].unique()

array(['Card', 'Cash', 'Upi', 'Cod'], dtype=object)

In [18]:
Q1_sales=df_urban['Sales'].quantile(0.25)
Q3_sales=df_urban['Sales'].quantile(0.75)

IQR_sales=Q3_sales-Q1_sales

Lower_sales=Q1_sales-1.5*IQR_sales
Upper_sales=Q3_sales+1.5*IQR_sales

Outer_sales=df_urban[(df_urban['Sales']<Lower_sales) | (df_urban['Sales']>Upper_sales)]

print(len(Outer_sales))

179


In [23]:
Q1_Profit=df_urban['Profit'].quantile(0.25)
Q3_Profit=df_urban['Profit'].quantile(0.75)

IQR_Profit=Q3_Profit-Q1_Profit

Lower_Profit=Q1_Profit-1.5*IQR_Profit
Upper_Profit=Q3_Profit+1.5*IQR_Profit

Outer_Profit=df_urban[(df_urban['Profit']<Lower_Profit) | (df_urban['Profit']>Upper_Profit)]

print(len(Outer_Profit))

161


In [28]:
sales_shew=df_urban['Sales'].skew()

print(sales_shew)

19.70106414003683


In [29]:
profit_skew=df_urban['Profit'].skew()

print(profit_skew)

-12.2245898037512


In [32]:
mean_sales=df_urban['Sales'].mean()

median_sales=df_urban['Sales'].median()

print("mean: ",mean_sales)
print("median: ",median_sales)


mean:  200.79307500000002
median:  86.73500000000001


In [33]:
mean_Profit=df_urban['Profit'].mean()

median_Profit=df_urban['Profit'].median()

print("mean: ",mean_Profit)
print("median: ",median_Profit)

mean:  4.536569999999999
median:  7.045


In [34]:
std_sales=df_urban['Sales'].std()

print(std_sales)

631.0158600813709


In [36]:
coefficient_sales=(std_sales/mean_sales)*100

print(coefficient_sales)

314.2617642970859


In [37]:
import numpy as np

len_profit=len(df_urban['Profit'])
Std_profit=df_urban['Profit'].std()
SE=Std_profit/np.sqrt(len_profit)

CI=mean_Profit-1.96*SE,mean_Profit+1.96*SE

print(CI)


(np.float64(-0.618919895392434), np.float64(9.692059895392433))


In [38]:
coor_matrix=df_urban[['Sales', 'Profit', 'Discount', 'Delivery_Days', 'Customer_Rating']].corr()

print(coor_matrix)

                    Sales    Profit  Discount  Delivery_Days  Customer_Rating
Sales            1.000000  0.087830  0.017906       0.028777        -0.056095
Profit           0.087830  1.000000 -0.251677      -0.024161         0.041752
Discount         0.017906 -0.251677  1.000000       0.021024        -0.007551
Delivery_Days    0.028777 -0.024161  0.021024       1.000000        -0.605624
Customer_Rating -0.056095  0.041752 -0.007551      -0.605624         1.000000


In [39]:
df_urban['Sales'].corr(df_urban['Discount'])

np.float64(0.017905840655899334)

In [40]:
df_urban['Sales'].corr(df_urban['Profit'])

np.float64(0.08783031734726718)

In [41]:
df_urban['Delivery_Days'].corr(df_urban['Customer_Rating'])

np.float64(-0.6056239938456098)

In [43]:
region_group = df_urban.groupby('Region').agg(
    Mean_sales=('Sales','mean'),
    Median_sales=('Sales','median'),
    Mean_profit=('Profit','mean'),
    Total_orders=('Order_ID','count')
)

region_group

,Mean_sales,Median_sales,Mean_profit,Total_orders
Region,,,,
East,195.709530,83.05,4.219983,574
North,232.838354,91.68,5.599911,559
South,194.570382,87.36,6.403194,576
West,161.579725,81.71,-0.576357,291


In [44]:
category_group = df_urban.groupby('Category').agg(
    Mean_sales=('Sales','mean'),
    Median_sales=('Sales','median'),
    Mean_profit=('Profit','mean'),
    Total_orders=('Order_ID','count')
)

category_group

,Mean_sales,Median_sales,Mean_profit,Total_orders
Category,,,,
Beauty,171.287439,87.385,1.039512,410
Books,225.036554,91.680,12.185012,415
Electronics,207.866789,80.095,9.552658,380
Fashion,178.534212,92.740,-3.057494,387
Home & Kitchen,220.308848,81.245,2.802451,408


In [45]:
segment_group = df_urban.groupby('Customer_Segment').agg(
    Mean_sales=('Sales','mean'),
    Median_sales=('Sales','median'),
    Mean_profit=('Profit','mean'),
    Total_orders=('Order_ID','count')
)

segment_group

,Mean_sales,Median_sales,Mean_profit,Total_orders
Customer_Segment,,,,
Consumer,198.372747,83.50,2.199636,659
Corporate,191.549054,86.21,0.240726,634
Home Office,211.338628,88.93,10.567129,707


In [46]:
globle_mean_sales=df_urban['Sales'].mean()

globle_mean_profit=df_urban['Profit'].mean()

print("Global Mean Sales: ",globle_mean_sales)
print("Global Mean Profit: ",globle_mean_profit)

Global Mean Sales:  200.79307500000002
Global Mean Profit:  4.536569999999999


In [47]:
region_group['sales_globle']=region_group['Mean_sales']-globle_mean_sales
region_group['profit_globle']=region_group['Mean_profit']-globle_mean_profit

region_group
#

,Mean_sales,Median_sales,Mean_profit,Total_orders,sales_globle,profit_globle
Region,,,,,,
East,195.709530,83.05,4.219983,574,-5.083545,-0.316587
North,232.838354,91.68,5.599911,559,32.045279,1.063341
South,194.570382,87.36,6.403194,576,-6.222693,1.866624
West,161.579725,81.71,-0.576357,291,-39.213350,-5.112927


In [48]:
category_group['sales_globle']=category_group['Mean_sales']-globle_mean_sales
category_group['profit_globle']=category_group['Mean_profit']-globle_mean_profit

category_group


,Mean_sales,Median_sales,Mean_profit,Total_orders,sales_globle,profit_globle
Category,,,,,,
Beauty,171.287439,87.385,1.039512,410,-29.505636,-3.497058
Books,225.036554,91.680,12.185012,415,24.243479,7.648442
Electronics,207.866789,80.095,9.552658,380,7.073714,5.016088
Fashion,178.534212,92.740,-3.057494,387,-22.258863,-7.594064
Home & Kitchen,220.308848,81.245,2.802451,408,19.515773,-1.734119


In [50]:
segment_group['sales_globle']=segment_group['Mean_sales']-globle_mean_sales
segment_group['profit_globle']=segment_group['Mean_profit']-globle_mean_profit

segment_group

,Mean_sales,Median_sales,Mean_profit,Total_orders,sales_globle,profit_globle
Customer_Segment,,,,,,
Consumer,198.372747,83.50,2.199636,659,-2.420328,-2.336934
Corporate,191.549054,86.21,0.240726,634,-9.244021,-4.295844
Home Office,211.338628,88.93,10.567129,707,10.545553,6.030559


In [51]:
region_over=region_group[
    (region_group['sales_globle']>globle_mean_sales) &
    (region_group['profit_globle']>globle_mean_profit)
]

region_under=region_group[
    (region_group['sales_globle']<globle_mean_sales) &
    (region_group['profit_globle']<globle_mean_profit)
]

print("Region Overperforming:")
print(region_over)

print("\nRegion Underperforming:")
print(region_under)

Region Overperforming:
Empty DataFrame
Columns: [Mean_sales, Median_sales, Mean_profit, Total_orders, sales_globle, profit_globle]
Index: []

Region Underperforming:
        Mean_sales  Median_sales  Mean_profit  Total_orders  sales_globle  \
Region                                                                      
East    195.709530         83.05     4.219983           574     -5.083545   
North   232.838354         91.68     5.599911           559     32.045279   
South   194.570382         87.36     6.403194           576     -6.222693   
West    161.579725         81.71    -0.576357           291    -39.213350   

        profit_globle  
Region                 
East        -0.316587  
North        1.063341  
South        1.866624  
West        -5.112927  


In [52]:
category_over=category_group[
    (category_group['sales_globle']>globle_mean_sales) &
    (category_group['profit_globle']>globle_mean_profit)]

category_under=category_group[
    (category_group['sales_globle']<globle_mean_sales) &
    (category_group['profit_globle']<globle_mean_profit)]

print("Category Overperforming:")
print(category_over)

print("\nCategory Underperforming:")
print(category_under)


Category Overperforming:
Empty DataFrame
Columns: [Mean_sales, Median_sales, Mean_profit, Total_orders, sales_globle, profit_globle]
Index: []

Category Underperforming:
                Mean_sales  Median_sales  Mean_profit  Total_orders  \
Category                                                              
Beauty          171.287439        87.385     1.039512           410   
Fashion         178.534212        92.740    -3.057494           387   
Home & Kitchen  220.308848        81.245     2.802451           408   

                sales_globle  profit_globle  
Category                                     
Beauty            -29.505636      -3.497058  
Fashion           -22.258863      -7.594064  
Home & Kitchen     19.515773      -1.734119  


In [54]:
segment_over=segment_group[
    (segment_group['sales_globle']<globle_mean_sales)&
    (segment_group['profit_globle']>globle_mean_profit)]

segment_under=segment_group[
    (segment_group['sales_globle']>globle_mean_sales)&
    (segment_group['profit_globle']<globle_mean_profit)]

print("Segment Overperforming:")
print(segment_over)

print("\nSegment Underperforming:")
print(segment_under)


Segment Overperforming:
                  Mean_sales  Median_sales  Mean_profit  Total_orders  \
Customer_Segment                                                        
Home Office       211.338628         88.93    10.567129           707   

                  sales_globle  profit_globle  
Customer_Segment                               
Home Office          10.545553       6.030559  

Segment Underperforming:
Empty DataFrame
Columns: [Mean_sales, Median_sales, Mean_profit, Total_orders, sales_globle, profit_globle]
Index: []


In [57]:
best_region = region_group['Mean_profit'].idxmax()
worst_region = region_group['Mean_profit'].idxmin()

print("Best Region:", best_region)
print("Worst Region:", worst_region)

Best Region: South
Worst Region: West


In [58]:
best_category = category_group['Mean_profit'].idxmax()
worst_category = category_group['Mean_profit'].idxmin()

print("Best Category:", best_category)
print("Worst Category:", worst_category)

Best Category: Books
Worst Category: Fashion


In [59]:
best_segment = segment_group['Mean_profit'].idxmax()
worst_segment = segment_group['Mean_profit'].idxmin()

print("Best Segment:", best_segment)
print("Worst Segment:", worst_segment)
#

Best Segment: Home Office
Worst Segment: Corporate
